### Calibration of wave gauge with temperature compensation ###

In [1]:
import logging
import sys
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional, Dict
from IPython.display import  display, HTML
plt.rcParams["figure.figsize"] = [6.5, 14]
plt.rcParams["figure.autolayout"] = True

from utils.init import LoggingStyleAdapter

pd.options.plotting.backend = 'holoviews'  # turns .plot to .plplot

# Logging
logger = logging.getLogger(__name__)
logger.handlers[:] = []
lf = LoggingStyleAdapter(logger)
# logger.setLevel(logging.DEBUG)
# Add one of handlers
if False:
    # Notebook DisplayHandler
    class DisplayHandler(logging.Handler):
        def emit(self, record):
            message = self.format(record)
            display(message)

    class HTMLFormatter(logging.Formatter):
        level_colors = {
            logging.DEBUG: 'lightblue',
            logging.INFO: 'dodgerblue',
            logging.WARNING: 'goldenrod',
            logging.ERROR: 'crimson',
            logging.CRITICAL: 'firebrick'
        }

        def __init__(self):
            super().__init__(
                '<span style="font-weight: bold; color: green">{asctime}</span> '
                '[<span style="font-weight: bold; color: {levelcolor}">{levelname}</span>] '
                '{message}',
                style='{'
            )

        def format(self, record):
            record.levelcolor = self.level_colors.get(record.levelno, 'black')
            return HTML(super().format(record))

    handler = DisplayHandler()
    handler.setFormatter(HTMLFormatter())

    logger.addHandler(handler)
else:
    # StreamHandler
    from colorama import Fore, Back, Style

    class ColoredFormatter(logging.Formatter):
        """Colored log formatter."""

        def __init__(self, *args, colors: Optional[Dict[str, str]]=None, **kwargs) -> None:
            """Initialize the formatter with specified format strings."""

            super().__init__(*args, **kwargs)

            self.colors = colors if colors else {}

        def format(self, record) -> str:
            """Format the specified record as text."""

            record.color = self.colors.get(record.levelname, '')
            record.reset = Style.RESET_ALL

            return super().format(record)


    formatter = ColoredFormatter(
        '{asctime} |{color} {levelname:8} {reset}| {name} | {message}',
        style='{', datefmt='%Y-%m-%d %H:%M:%S',
        colors={
            'DEBUG': Fore.CYAN,
            'INFO': Fore.GREEN,
            'WARNING': Fore.YELLOW,
            'ERROR': Fore.RED,
            'CRITICAL': Fore.RED + Back.WHITE + Style.BRIGHT,
        }
    )

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(formatter)
    logger.addHandler(handler)


In [2]:
# Load data
# %tb
# Custom importing function
# importing my functions
# drive_d = 'D:' if sys.platform == 'win32' else '/mnt/D'  # to run on my Linux/Windows systems both
# scripts_path = Path(drive_d + '/Work/_Python3/And0K/h5toGrid/inclinometer')
# sys.path.append(str(Path(scripts_path).resolve()))
from csv_load import load_from_csv_gen
cfg_in = {'text_path':
    Path(r'd:\WorkData\_experiment\inclinometer\230425_Грузопоршневой\_raw\P\INKL_P{number:0>2}*.TXT')
}
for i, probe_id, path_csv, df_raw in load_from_csv_gen(cfg_in):
    lf.info('loaded {}', i)
    break
df_raw = df_raw[['P_counts', 'Temp']]

Raw files for 5 probes found:
p01: [WindowsPath('d:/WorkData/_experiment/inclinometer/230425_Грузопоршневой/_raw/P/@incl_p01.TXT')]
p02: [WindowsPath('d:/WorkData/_experiment/inclinometer/230425_Грузопоршневой/_raw/P/@incl_p02.TXT')]
p03: [WindowsPath('d:/WorkData/_experiment/inclinometer/230425_Грузопоршневой/_raw/P/@incl_p03.TXT')]
p04: [WindowsPath('d:/WorkData/_experiment/inclinometer/230425_Грузопоршневой/_raw/P/@incl_p04.TXT')]
p05: [WindowsPath('d:/WorkData/_experiment/inclinometer/230425_Грузопоршневой/_raw/P/@incl_p05.TXT')]
Loading 5 raw files...


79.9 % non-increased...


D:\Work\_Python3\And0K\h5toGrid\filters.py:57: NumbaExperimentalFeatureWarning: Use of isinstance() detected. This is an experimental feature.
  return np.where(bOk if bOk is not None else b, y[b], np.nan)


Linearise time using reference frequency 5.0 (between holes > 2.0 s)
.2023-06-17 21:09:27 | INFO     | __main__ | loaded 1


In [3]:
lf.info('Processing file #{}. Data:', i)
df_raw

2023-06-17 21:09:47 | INFO     | __main__ | Processing file #1. Data:


,P_counts,Temp
2023-04-25 12:24:11+00:00,1849841.0,21.63
2023-04-25 12:24:11.200000+00:00,1849809.0,21.63
2023-04-25 12:24:11.400000+00:00,1849805.0,21.63
2023-04-25 12:24:12+00:00,1849720.0,21.63
2023-04-25 12:24:12.200000+00:00,1849746.0,21.63
...,...,...
2023-04-25 18:44:40.600000+00:00,1836966.0,28.61
2023-04-25 18:44:40.800000+00:00,1836977.0,28.62
2023-04-25 18:44:41+00:00,1836888.0,28.62
2023-04-25 18:44:41.200000+00:00,1836927.0,28.62


In [14]:
import panel as pn
from bokeh.plotting import figure
# pn.extension()
pn.config.sizing_mode = 'stretch_width'
pn.config.

In [21]:
fig = figure()  # width=300, height=300)
xs = np.linspace(0, 10)
r = p.line(xs, np.sin(xs))

width_slider = pn.widgets.FloatSlider(name='Line Width', start=0.1, end=10)
width_slider.jslink(r.glyph, value='line_width')

layout = pn.Column(width_slider, fig)
# layout = pn.Row(text_input, markdown)


srv = pn.serve(layout, show=True)

INFO:bokeh.server.server:Starting Bokeh server version 3.1.1 (running on Tornado 6.3.2)
INFO:bokeh.server.tornado:User authentication hooks NOT provided (default user enabled)


Launching server at http://localhost:14055


In [12]:
width_slider.value

'Some text1245'

In [22]:
# srv.stop()
pn.state.kill_all_servers()

In [70]:
if False:
    style = """
    <style>
    div.output_area {
        overflow-y: scroll;
    }
    div.output_area img {
        max-width: unset;
    }
    </style>
    """
    HTML(style)


# def mouse_event(event):
#     print('x: {} and y: {}'.format(event.xdata, event.ydata))

#fig = plt.figure()
#cid = fig.canvas.mpl_connect('button_press_event', mouse_event)

df_raw.plot(grid=True, xlabel='Time')  # , include=['P_counts', 'Temp'] secondary_y = df_raw[['P_counts', 'Temp']]

# plt.plot(x, y)
# plt.show()
# click anywhere on the plot, and it will show the coordinates of the points on the console

AttributeError: Selection.selected property descriptor does not exist